In [1]:
import sys
sys.path.append('/root/autodl-tmp/CommitFit')
import pandas as pd
from sklearn.model_selection import train_test_split
import ensemble_model.preprocesser as preprocesser 
import ensemble_model.MoE_model as moe 
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer, RobertaModel, RobertaTokenizer
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve

2026-02-21 19:58:04.229792: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-21 19:58:04.283570: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-21 19:58:05.287640: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/root/miniconda3/lib

In [2]:
# =========================
# 0) 配置
# =========================
CSV_PATH = "generated_commits.csv"

# =========================
# 2) 读 CSV -> Dataset -> split
# =========================
df = pd.read_csv(CSV_PATH)
label2id={'Adaptive':0, 'Corrective':1, 'Perfective':2}
df = df.replace({"labels": label2id})
df




# df["diff_compact"] = df["diffs"].map(compress_diff_minimal)
# df["gen_prompt"] = df["diff_compact"].map(lambda s: "generate commit message: " + s)

# df = df.rename(columns={'msgs':'target_text','gen_prompt':'source_text'})
label2id={'Adaptive':0, 'Corrective':1, 'Perfective':2}
df = df.replace({"labels": label2id})
df
df.dropna(inplace=True)

In [3]:
train, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(temp_df, test_size=0.5, random_state=42)

In [4]:
train['source_text'].str.len()

1452    3942
762      738
932      885
435      609
629     1452
        ... 
1130    1305
1294    2071
860      975
1459    2948
1126     786
Name: source_text, Length: 1022, dtype: int64

In [5]:
# train['msgs'][1497]

In [6]:
# train = train.groupby("label", group_keys=False).apply(lambda x: x.sample(n=10, random_state=42))

In [7]:
# Load BERT and CodeBERT models and tokenizers
bert_model = BertModel.from_pretrained('/root/autodl-tmp/models/google-bert/bert-base-uncased')
bert_tokenizer = BertTokenizer.from_pretrained('/root/autodl-tmp/models/google-bert/bert-base-uncased')

codebert_model = RobertaModel.from_pretrained('/root/autodl-tmp/models/codebert-base')
codebert_tokenizer = RobertaTokenizer.from_pretrained('/root/autodl-tmp/models/codebert-base')

In [8]:
train

,user,repo,commit,labels,msgs,diffs,feature,target_text,changescribe_text,core_diff,source_text,pred
1452,apache,hama,caeeaaa03f71fffaba3a03409b724e4cd9fc03e2,2,HAMA-783: Efficient InMemory Storage for Verti...,diff --git a/examples/src/main/java/org/apache...,"[10, 72, 42, 40, 6, 0, 1, 2, 5, 5, 4, 0, 0, 0,...",Efficient InMemory Storage for Vertices,hama [Change] ChangeScribeStart\nSummarized Co...,Summarized Code Changes:\nFile: hama/examples/...,You are an experienced software engineer writi...,Efficient InMemory Storage for Vertices
762,GNOME,valadoc,935820dbb5b80a81e66f432ec5199516bc98e998,0,driver/0.11.0: Add support for attributes\n,diff --git a/src/driver/0.11.0/treebuilder.val...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",driver/0.11.0: Add support for attributes,valadoc [Change] ChangeScribeStart\nSummarized...,Summarized Code Changes:\nFile: driver/0.11.0/...,You are an experienced software engineer writi...,driver/0.11.0: Add support for attributes
932,eclipse-mylyn,org.eclipse.mylyn.reviews,8d62686d2a5e00f381a2b333b05c4fd70bf8ffb4,0,"322734: changed labels, border for rating comm...",diff --git a/org.eclipse.mylyn.reviews.ui/src/...,"[2, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","322734: changed labels, border for rating comment",org.eclipse.mylyn.reviews [Change] ChangeScrib...,Summarized Code Changes:\nFile: ui/editors/Rev...,You are an experienced software engineer writi...,"322734: changed labels, border for rating comment"
435,restlet,restlet-framework-java,0f8ab8e6e1bdac17585abf685efda33d71cbbff1,1,- The ServletContextAdapter passed by the- Se...,diff --git a/modules/com.noelios.restlet.ext.s...,"[1, 4, 5, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",The ServletContextAdapter passed by the Server...,restlet-framework-java [Change] ChangeScribeSt...,Summarized Code Changes:\nFile: ext/servlet/Se...,You are an experienced software engineer writi...,The ServletContextAdapter passed by the Server...
629,spring-projects,spring-framework,59d80ec19e7e6333a2af64b98f68d3efecc58a97,1,Fix minor issue in MockHttpServletRequest--Pre...,diff --git a/spring-test/src/main/java/org/spr...,"[0, 0, 0, 0, 0, 0, 0, 2, 38, 8, 22, 4, 0, 5, 0...",minor issue in MockHttpServletRequest Previous...,spring-framework [Change] ChangeScribeStart\nS...,Summarized Code Changes:\nFile: mock/web/MockH...,You are an experienced software engineer writi...,minor issue in MockHttpServletRequest Issue: -
...,...,...,...,...,...,...,...,...,...,...,...,...
1130,GNOME,vala,2c95c42e94bb4cb51d2bc9e09f09686a6adf162d,1,libsoup-2.4: fix arguments to several methods ...,diff --git a/vapi/libsoup-2.4.vapi b/vapi/libs...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",libsoup-2.4: arguments to several methods whic...,vala [Change] ChangeScribeStart\nSummarized Co...,Summarized Code Changes:\nFile: vapi/libsoup-2...,You are an experienced software engineer writi...,libsoup-2.4: arguments to multiple methods
1294,arrayexpress,annotare2,011627c0f38cfbc5b88858f8641aefc55b5df58a,2,Added some flexibility with scan protocol assi...,diff --git a/app/om/src/main/java/uk/ac/ebi/fg...,"[2, 33, 5, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Added some flexibility with scan protocol assi...,annotare2 [Change] ChangeScribeStart\nSummariz...,Summarized Code Changes:\nFile: submission/mod...,You are an experienced software engineer writi...,Added some flexibility with scan protocol assi...
860,tapiji,tapiji,02e52f66e05400002b043794d7fcd9d165239c2a,2,increases generation performace of auto-comple...,diff --git a/org.eclipse.babel.tapiji.tools.ja...,"[1, 11, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",increases generation performace of auto-comple...,tapiji [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/ui/Messag...,You are an experienced software engineer writi...,increases generation performace of auto-comple...
1459,casidiablo,persistence,d7bf95159df37a3d338ca267dddd3d26b38ec37c,2,Now it is possible to specify the sqlite open ...,

In [9]:
train.reset_index(inplace=True)
val.reset_index(inplace=True)
test.reset_index(inplace=True)

# Create Datasets and DataLoaders
train_dataset = preprocesser.SentencePairDataset(train, bert_tokenizer, codebert_tokenizer,message='pred',command='changescribe_text',label='labels')
val_dataset = preprocesser.SentencePairDataset(val, bert_tokenizer, codebert_tokenizer,message='pred',command='changescribe_text',label='labels')
test_dataset = preprocesser.SentencePairDataset(test, bert_tokenizer, codebert_tokenizer,message='pred',command='changescribe_text',label='labels')
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



# # Initialize the model
# model = em.CombinedModel(bert_model, codebert_model, bert_tokenizer, codebert_tokenizer)

# # Train the model
# model.trainer(train_loader, val_loader,num_epochs=10)
# for batch in train_loader:
#     print(batch)


base_model1 = moe.BaseModel(bert_model)
base_model2 = moe.BaseModel(codebert_model)

# Create stacking model
combined_model = moe.MoEModel(base_model1, base_model2)
# Train the model
combined_model.trainer(train_loader, val_loader,num_epochs=10, patience=3)

Epoch 1/10 Loss: 1.0899: 100%|██████████| 32/32 [00:16<00:00,  1.90batch/s]


=============================train========================


Epoch 2/10 Loss: 1.0099: 100%|██████████| 32/32 [00:16<00:00,  2.00batch/s]


=============================train========================


Epoch 3/10 Loss: 0.8917: 100%|██████████| 32/32 [00:16<00:00,  1.99batch/s]


...

=============================train========================


Epoch 9/10 Loss: 0.3313: 100%|██████████| 32/32 [00:16<00:00,  1.95batch/s]


=============================train========================


Epoch 10/10 Loss: 0.2355: 100%|██████████| 32/32 [00:16<00:00,  1.95batch/s]

=============================train========================


In [10]:
test_dataset = preprocesser.SentencePairDataset(test, bert_tokenizer, codebert_tokenizer,message='msgs',command='changescribe_text',label='labels')
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [11]:
test_acc, test_labels, test_probabilities, test_embeddings, test_predictions = combined_model.evaluate(test_loader)

Validation Accuracy: 0.6455
Precision: 0.6477
Recall: 0.6455
F1-Score: 0.6448


In [12]:
from sklearn import metrics

In [13]:
cm = metrics.confusion_matrix(test_labels, test_predictions)
cm

array([[62, 13, 14],
       [15, 39,  7],
       [23,  6, 41]])

In [14]:
import torch
# 保存
torch.save(
    {
        "model_state_dict": combined_model.state_dict(),
        "num_labels": 3,
    },
    "./reward_model_combined-3.pt",
)

# 加载
ckpt = torch.load("reward_model_combined-3.pt")
combined_model.load_state_dict(ckpt["model_state_dict"])

<All keys matched successfully>

In [15]:
# 加载
# reward_model = torch.load("reward_model_combined.pth")
# combined_model.load_state_dict(ckpt["model_state_dict"])

In [16]:
#发送多种类型的邮件
from email.mime.multipart import MIMEMultipart
import smtplib

from email.mime.text import MIMEText
msg_from = '915803745@qq.com'  # 发送方邮箱
passwd = 'vcuosuurrgkfbdai'   #就是上面的授权码
 
# to= ['g.zhang@gotion.com', 'j.tong@gotion.com'] #接受方邮箱
to= ['j.tong@gotion.com'] #接受方邮箱
#设置邮件内容
#MIMEMultipart类可以放任何内容
msg = MIMEMultipart()
conntent=f"强化学习"
#把内容加进去
msg.attach(MIMEText(conntent,'plain','utf-8'))
 
#设置邮件主题
msg['Subject']="强化学习模型训练完毕"
 
#发送方信息
msg['From']=msg_from
 
#开始发送
 
#通过SSL方式发送，服务器地址和端口
s = smtplib.SMTP_SSL("smtp.qq.com", 465)
# 登录邮箱
s.login(msg_from, passwd)
#开始发送
s.sendmail(msg_from,to,msg.as_string())
print("强化学习模型训练完毕")

强化学习模型训练完毕
